# LeetCode 532 — K-diff Pairs in an Array bằng Python

Link đề: https://leetcode.com/problems/k-diff-pairs-in-an-array/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, Two Pointers, Binary Search, Sorting

Skeleton LeetCode:

```python
class Solution:
    def findPairs(self, nums: List[int], k: int) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: đếm số **cặp giá trị phân biệt** `(a, b)` trong mảng sao cho `|a - b| == k`, trong đó hai phần tử phải ở **hai vị trí khác nhau** (`i != j`) và các cặp **giống nhau về giá trị** chỉ được tính **một lần**.
2. Nắm các công cụ Python: `set`, `collections.Counter`, `sorted`/`list.sort`, `bisect`, `abs`, tuple làm khóa băm, kỹ thuật hai con trỏ.
3. Đi từ **vét cạn mọi cặp chỉ số + `set` các cặp** → **một lượt với hai `set` (đã thấy / đáp án)** → **`Counter` (tách riêng `k == 0` và `k > 0`)** → **sắp xếp + hai con trỏ** → **sắp xếp + tìm kiếm nhị phân**.
4. Hiểu mấu chốt: cặp được xác định bởi **giá trị nhỏ hơn** (vì giá trị lớn hơn `= nhỏ + k`), nên chỉ cần đếm **các giá trị `x` sao cho `x + k` cũng có trong mảng**; trường hợp đặc biệt `k = 0` cần `x` xuất hiện **ít nhất hai lần**.
5. Tránh các lỗi kinh điển: đếm trùng cặp, quên `k = 0` (một phần tử đứng một mình không tạo thành cặp với chính nó), đếm mỗi cặp hai lần khi xét cả `x - k` lẫn `x + k`, và quên bỏ qua giá trị lặp trong hai con trỏ.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums` và số nguyên `k`. Hãy trả về **số cặp k-diff duy nhất (unique)** trong mảng.

Một cặp **k-diff** là cặp số nguyên `(nums[i], nums[j])` thỏa mãn đồng thời:

- `0 <= i, j < nums.length`
- `i != j`
- `|nums[i] - nums[j]| == k`

(`|val|` là giá trị tuyệt đối.) "Duy nhất" nghĩa là hai cặp có **cùng cặp giá trị** chỉ tính một lần, không quan tâm chúng nằm ở chỉ số nào, và không quan tâm thứ tự (`(1, 3)` và `(3, 1)` là một cặp).

**Ví dụ 1**

- Input: `nums = [3,1,4,1,5], k = 2`
- Output: `2`
- Giải thích: có hai cặp 2-diff là `(1, 3)` và `(3, 5)`. Dù có hai số `1` trong mảng, cặp `(1, 3)` vẫn chỉ tính một lần.

**Ví dụ 2**

- Input: `nums = [1,2,3,4,5], k = 1`
- Output: `4`
- Giải thích: các cặp 1-diff là `(1,2)`, `(2,3)`, `(3,4)`, `(4,5)`.

**Ví dụ 3**

- Input: `nums = [1,3,1,5,4], k = 0`
- Output: `1`
- Giải thích: chỉ có một cặp 0-diff là `(1, 1)` (hai số `1` ở hai vị trí khác nhau).

**Ràng buộc:**

- `1 <= nums.length <= 10^4`
- `-10^7 <= nums[i] <= 10^7`
- `0 <= k <= 10^7`

Chú ý: `k` có thể bằng `0`, và mảng có thể có giá trị lặp, có số âm, chỉ có một phần tử.

In [1]:
from typing import List
from collections import Counter
from itertools import combinations
from bisect import bisect_left
import random
import time

examples = [
    ([3, 1, 4, 1, 5], 2, 2),
    ([1, 2, 3, 4, 5], 1, 4),
    ([1, 3, 1, 5, 4], 0, 1),
]
for nums, k, expected in examples:
    print(f"nums = {nums}, k = {k} -> kỳ vọng {expected}")

nums = [3, 1, 4, 1, 5], k = 2 -> kỳ vọng 2
nums = [1, 2, 3, 4, 5], k = 1 -> kỳ vọng 4
nums = [1, 3, 1, 5, 4], k = 0 -> kỳ vọng 1


## 3. Kiến thức nền

### 3.1. Tuple và `set` — cách lưu "một cặp chỉ một lần"

`set` là tập hợp **không chứa phần tử trùng**; thêm hai lần cùng một phần tử thì vẫn chỉ có một. Tuple như `(1, 3)` **băm được** nên đưa vào `set` được (còn `list` thì không). Kiểm tra `x in s` trung bình `O(1)`.

In [2]:
s = set()
s.add((1, 3))
s.add((1, 3))          # trùng -> không thêm
s.add((3, 5))
print(s, "| số phần tử:", len(s))
print((1, 3) in s, (3, 1) in s)    # thứ tự trong tuple có ý nghĩa

{(1, 3), (3, 5)} | số phần tử: 2
True False


Vì `(1, 3)` và `(3, 1)` là **hai tuple khác nhau**, ta phải **chuẩn hóa**: luôn viết giá trị **nhỏ trước, lớn sau** (`(min(a, b), max(a, b))`) thì mỗi cặp có đúng một dạng.

### 3.2. Giá trị tuyệt đối và `|a - b| == k`

`abs(x)` là giá trị tuyệt đối. Với `k >= 0`: `|a - b| == k` ⇔ `a - b == k` **hoặc** `b - a == k`; tức là hai số hơn kém nhau đúng `k`. Nếu gọi `a` là số nhỏ, `b = a + k` là số lớn.

In [3]:
print(abs(3 - 1), abs(1 - 3), abs(5 - 5))
a, b = 1, 3
print("nhỏ:", min(a, b), "| lớn:", max(a, b), "| lớn - nhỏ =", max(a, b) - min(a, b))
print("|a-b| == 2 ?", abs(a - b) == 2)

2 2 0
nhỏ: 1 | lớn: 3 | lớn - nhỏ = 2
|a-b| == 2 ? True


### 3.3. `Counter` — đếm tần suất

`collections.Counter(nums)` cho biết mỗi giá trị xuất hiện bao nhiêu lần. Nó cũng là một `dict`: duyệt qua các **khóa phân biệt** bằng `for x in cnt`, tra tần suất bằng `cnt[x]`; khóa vắng mặt trả về `0` (không lỗi).

In [4]:
cnt = Counter([3, 1, 4, 1, 5])
print(cnt)
print("các giá trị phân biệt:", list(cnt))
print("cnt[1] =", cnt[1], "| cnt[9] (vắng) =", cnt[9])
print("1 in cnt:", 1 in cnt, "| 9 in cnt:", 9 in cnt)

Counter({1: 2, 3: 1, 4: 1, 5: 1})
các giá trị phân biệt: [3, 1, 4, 5]
cnt[1] = 2 | cnt[9] (vắng) = 0
1 in cnt: True | 9 in cnt: False


Lưu ý: `cnt[9]` chỉ **đọc** nên không thêm khóa `9` vào `Counter`; nhưng `9 in cnt` mới là cách kiểm tra "có mặt hay không" rõ ràng.

### 3.4. Sắp xếp và nhóm giá trị trùng

Sau khi sắp xếp, các giá trị bằng nhau đứng **liền nhau**. Nhờ vậy ta bỏ qua trùng chỉ bằng cách so với phần tử ngay trước.

In [5]:
nums = [3, 1, 4, 1, 5]
sorted_nums = sorted(nums)          # sorted trả list mới, không phá nums
print(sorted_nums)
for i, x in enumerate(sorted_nums):
    if i > 0 and x == sorted_nums[i - 1]:
        print(f"vị trí {i}: {x} trùng với phần tử trước -> bỏ qua")
    else:
        print(f"vị trí {i}: {x} là giá trị mới")

[1, 1, 3, 4, 5]
vị trí 0: 1 là giá trị mới
vị trí 1: 1 trùng với phần tử trước -> bỏ qua
vị trí 2: 3 là giá trị mới
vị trí 3: 4 là giá trị mới
vị trí 4: 5 là giá trị mới


### 3.5. Hai con trỏ trên mảng đã sắp xếp

Khi mảng tăng dần, nếu `nums[j] - nums[i]` **còn nhỏ hơn mục tiêu** thì ta tăng `j` (để hiệu lớn lên); nếu **lớn hơn** thì tăng `i`. Cả hai con trỏ **chỉ đi tới**, không quay lại, nên tổng cộng tốn `O(n)` bước. Đây là kỹ thuật quen thuộc của bài Two Sum trên mảng đã sắp xếp.

### 3.6. `bisect_left` — tìm kiếm nhị phân có sẵn

`bisect_left(a, x, lo)` trả về chỉ số `p` nhỏ nhất (từ `lo` trở đi) sao cho `a[p] >= x` trên mảng đã sắp xếp, trong `O(log n)`. Muốn biết `x` có trong `a` không: kiểm tra `p < len(a) and a[p] == x`.

In [6]:
a = [1, 1, 3, 4, 5]
for x in (3, 2, 5, 6):
    p = bisect_left(a, x)
    found = p < len(a) and a[p] == x
    print(f"tìm {x}: bisect_left = {p}, có trong mảng? {found}")

# tham số lo: chỉ tìm từ vị trí lo trở đi
print("tìm 1 từ vị trí 2 trở đi:", bisect_left(a, 1, 2), "(vị trí 2 là chỗ chèn, a[2] = 3 != 1)")

tìm 3: bisect_left = 2, có trong mảng? True
tìm 2: bisect_left = 2, có trong mảng? False
tìm 5: bisect_left = 4, có trong mảng? True
tìm 6: bisect_left = 5, có trong mảng? False
tìm 1 từ vị trí 2 trở đi: 2 (vị trí 2 là chỗ chèn, a[2] = 3 != 1)


## 4. Ý tưởng nền tảng / trực giác (tính tay)

Lấy `nums = [3, 1, 4, 1, 5]`, `k = 2`. Hãy bỏ qua việc "cặp" là hai vị trí và nghĩ theo **giá trị**:

- Các **giá trị phân biệt**: `1, 3, 4, 5`.
- Một cặp 2-diff là hai giá trị hơn kém nhau 2. Mỗi cặp xác định bởi **giá trị nhỏ** `x` (giá trị lớn tự suy ra là `x + 2`).
- Thử từng giá trị nhỏ `x`: `x = 1` → cần `3` (có) ✓; `x = 3` → cần `5` (có) ✓; `x = 4` → cần `6` (không); `x = 5` → cần `7` (không).
- Đáp án = `2` (đó là `(1,3)` và `(3,5)`).

Nhận xét: số `1` xuất hiện hai lần nhưng vì ta đi theo **giá trị phân biệt**, cặp `(1,3)` tự nhiên chỉ xuất hiện **một lần**. Vậy bài toán = **đếm số giá trị phân biệt `x` sao cho `x + k` cũng có trong mảng**.

Ngoại lệ: **`k = 0`** thì `x + k = x` — chính giá trị `x`. Khi đó `x` phải xuất hiện **ít nhất hai lần** (hai vị trí khác nhau), nếu không `x` chỉ "ghép" với chính nó, điều đề cấm (`i != j`).

In [7]:
nums, k = [3, 1, 4, 1, 5], 2
distinct = sorted(set(nums))
print("giá trị phân biệt:", distinct)
for x in distinct:
    print(f"x = {x}: cần {x + k} -> {'có' if (x + k) in set(nums) else 'không'}")

# trường hợp k = 0: phải xuất hiện >= 2 lần
nums0 = [1, 3, 1, 5, 4]
cnt0 = Counter(nums0)
print("k = 0:", {x: c for x, c in cnt0.items()}, "-> các giá trị có tần suất >= 2:", [x for x, c in cnt0.items() if c >= 2])

giá trị phân biệt: [1, 3, 4, 5]
x = 1: cần 3 -> có
x = 3: cần 5 -> có
x = 4: cần 6 -> không
x = 5: cần 7 -> không
k = 0: {1: 2, 3: 1, 5: 1, 4: 1} -> các giá trị có tần suất >= 2: [1]


## 5. Cách 1: Brute Force (vét cạn)

### Ý tưởng

Làm đúng theo định nghĩa: thử **mọi cặp chỉ số** `i < j`; nếu `|nums[i] - nums[j]| == k` thì ghi nhận cặp giá trị đã chuẩn hóa `(min, max)` vào một `set`. `set` tự loại các cặp trùng. Đáp án là kích thước của `set`.

(Chỉ cần duyệt `i < j` vì cặp `(i, j)` và `(j, i)` cho cùng cặp giá trị.)

In [8]:
def pairs_brute(nums, k):
    seen_pairs = set()
    n = len(nums)
    for i in range(n):
        for j in range(i + 1, n):
            if abs(nums[i] - nums[j]) == k:
                a, b = nums[i], nums[j]
                seen_pairs.add((min(a, b), max(a, b)))
    return seen_pairs


def find_pairs_brute(nums, k):
    return len(pairs_brute(nums, k))


for nums, k, expected in examples:
    ps = pairs_brute(nums, k)
    print(sorted(ps), "->", len(ps), "(kỳ vọng", expected, ")")
    assert len(ps) == expected

[(1, 3), (3, 5)] -> 2 (kỳ vọng 2 )
[(1, 2), (2, 3), (3, 4), (4, 5)] -> 4 (kỳ vọng 4 )
[(1, 1)] -> 1 (kỳ vọng 1 )


### Độ phức tạp

- Có `n * (n - 1) / 2` cặp chỉ số. Với `n = 10^4` đó là gần **50 triệu** phép so sánh — mỗi phép trong Python tốn cỡ vài chục đến vài trăm nano giây ⇒ chậm (vài giây đến chục giây), dễ quá thời gian.
- Time `O(n^2)`; Space `O(p)` với `p` là số cặp phân biệt (tối đa `O(n)`).

### Nút thắt

Ta so sánh mọi cặp trong khi **với mỗi phần tử `x`, chỉ có đúng hai giá trị có thể ghép với nó** là `x - k` và `x + k`. Việc dò cả mảng để tìm hai giá trị đó là lãng phí; ta cần "tra" nhanh một giá trị có trong mảng hay không.

In [9]:
for n in (100, 1000, 10000):
    print(f"n = {n:>6}: số cặp chỉ số phải xét = {n * (n - 1) // 2:>12,}")

# đo thực tế với n nhỏ để thấy độ tăng bậc hai
rng = random.Random(1)
for n in (500, 1000, 2000):
    arr = [rng.randint(-10**7, 10**7) for _ in range(n)]
    t0 = time.perf_counter()
    find_pairs_brute(arr, 12345)
    print(f"n = {n:>5}: vét cạn mất {(time.perf_counter() - t0) * 1000:7.1f} ms")

n =    100: số cặp chỉ số phải xét =        4,950
n =   1000: số cặp chỉ số phải xét =      499,500
n =  10000: số cặp chỉ số phải xét =   49,995,000
n =   500: vét cạn mất     7.7 ms
n =  1000: vét cạn mất    30.4 ms


n =  2000: vét cạn mất   118.8 ms


Khi `n` gấp đôi thì thời gian gấp khoảng **bốn** lần — đặc trưng của `O(n^2)`.

## 6. Ý tưởng thuật toán / bước đột phá

Từ mục 4: một cặp được xác định bởi **giá trị nhỏ `x`**, giá trị lớn là `x + k`. Vậy:

> **Đáp án = số giá trị phân biệt `x` sao cho `x + k` cũng xuất hiện trong mảng** (với `k = 0` thì `x` phải xuất hiện ≥ 2 lần).

Để tra "`x + k` có trong mảng không" trong `O(1)`, dùng **bảng băm** (`set`/`Counter`). Hoặc sắp xếp rồi dùng hai con trỏ/tìm nhị phân. Tất cả đều tránh việc so từng cặp: thay `O(n^2)` bằng `O(n)` (băm) hoặc `O(n log n)` (sắp xếp).

Có bốn cách hiện thực ý tưởng "tra cứu thay vì dò":

1. **Một lượt + hai `set`**: duyệt từng số, tra `x - k` và `x + k` trong tập đã thấy.
2. **`Counter`**: đếm tần suất một lần rồi đếm trực tiếp các `x` thỏa điều kiện.
3. **Sắp xếp + hai con trỏ**: không cần bộ nhớ băm.
4. **Sắp xếp + tìm nhị phân**: với mỗi giá trị phân biệt, tìm `x + k`.

In [10]:
# minh họa nhận định: đáp án = số giá trị phân biệt x mà x + k cũng có mặt (k > 0)
for nums, k, expected in examples:
    if k > 0:
        s = set(nums)
        ans = sum(1 for x in s if x + k in s)
        print(nums, k, "->", ans, "(kỳ vọng", expected, ")")
        assert ans == expected

[3, 1, 4, 1, 5] 2 -> 2 (kỳ vọng 2 )
[1, 2, 3, 4, 5] 1 -> 4 (kỳ vọng 4 )


## 7. Các cách tiếp theo

### 7.1. Cách 2 — Một lượt với hai `set` (`vis` và `ans`)

**Giải thích từng đoạn:**

1. `vis` = các giá trị **đã duyệt** từ đầu mảng đến trước phần tử hiện tại. `ans` = tập các **giá trị nhỏ** của các cặp đã tìm thấy (mỗi cặp được nhận diện duy nhất bởi giá trị nhỏ).
2. Với số mới `x`, nó có thể ghép với một số **đã thấy** theo hai cách:
   - `x - k` đã thấy: cặp `(x - k, x)`, giá trị nhỏ là `x - k` → `ans.add(x - k)`.
   - `x + k` đã thấy: cặp `(x, x + k)`, giá trị nhỏ là `x` → `ans.add(x)`.
3. Sau khi xét, `vis.add(x)`. Vì mọi cặp gồm hai phần tử ở hai vị trí khác nhau, đến lúc phần tử **thứ hai** của cặp được duyệt thì phần tử đầu đã nằm trong `vis` → cặp luôn được phát hiện.
4. Với `k = 0`: `x - k = x + k = x`; chỉ khi `x` đã thấy (xuất hiện lần thứ hai trở lên) mới `ans.add(x)`; do `vis.add(x)` được làm **sau** cùng nên lần đầu thấy `x` không bị coi là cặp với chính nó.

**Code demo:**

In [11]:
def hashset_demo(nums, k):
    vis, ans = set(), set()
    for x in nums:
        if x - k in vis:
            ans.add(x - k)
            print(f"  x = {x}: thấy {x - k} trước đó -> cặp ({x - k}, {x}), ans = {sorted(ans)}")
        if x + k in vis:
            ans.add(x)
            print(f"  x = {x}: thấy {x + k} trước đó -> cặp ({x}, {x + k}), ans = {sorted(ans)}")
        vis.add(x)
    return len(ans)

print("đáp án:", hashset_demo([3, 1, 4, 1, 5], 2))
print("đáp án:", hashset_demo([1, 3, 1, 5, 4], 0))

  x = 1: thấy 3 trước đó -> cặp (1, 3), ans = [1]
  x = 1: thấy 3 trước đó -> cặp (1, 3), ans = [1]
  x = 5: thấy 3 trước đó -> cặp (3, 5), ans = [1, 3]
đáp án: 2
  x = 1: thấy 1 trước đó -> cặp (1, 1), ans = [1]
  x = 1: thấy 1 trước đó -> cặp (1, 1), ans = [1]
đáp án: 1


**Code nộp LeetCode (cách 2):**

In [12]:
class Solution2:
    def findPairs(self, nums: List[int], k: int) -> int:
        ans = set()
        vis = set()
        for x in nums:
            if x - k in vis:
                ans.add(x - k)
            if x + k in vis:
                ans.add(x)
            vis.add(x)
        return len(ans)


for nums, k, expected in examples:
    assert Solution2().findPairs(nums, k) == expected
print("Cách 2 OK")

Cách 2 OK


**Độ phức tạp:** Time `O(n)` (mỗi số làm hai phép tra `set` trung bình `O(1)`). Space `O(n)` cho hai `set`.

### 7.2. Cách 3 — `Counter` (tách riêng `k = 0` và `k > 0`)

**Ý tưởng khác:** đếm tần suất **trước**, rồi chỉ duyệt các **giá trị phân biệt**:

- `k == 0`: cặp `(x, x)` tồn tại khi `x` xuất hiện **≥ 2 lần** → đáp án = số giá trị có `cnt[x] >= 2`.
- `k > 0`: cặp `(x, x + k)` tồn tại khi `x + k` có trong `Counter` → đáp án = số giá trị `x` có `x + k in cnt`. (Số lần xuất hiện không quan trọng, chỉ cần có mặt.)

**Giải thích từng đoạn:**

1. `cnt = Counter(nums)`: một lượt đếm.
2. Nếu `k == 0`: `sum(1 for c in cnt.values() if c >= 2)`.
3. Ngược lại: `sum(1 for x in cnt if x + k in cnt)`. Mỗi cặp chỉ được đếm một lần vì nó gắn với giá trị **nhỏ** `x` (nếu đếm cả `x - k` thì mỗi cặp sẽ bị tính hai lần — đó là lỗi hay gặp).

**Code demo:**

In [13]:
def counter_demo(nums, k):
    cnt = Counter(nums)
    print("cnt =", dict(cnt))
    if k == 0:
        good = [x for x, c in cnt.items() if c >= 2]
        print("k = 0 -> các giá trị xuất hiện >= 2 lần:", good)
    else:
        good = [x for x in cnt if x + k in cnt]
        print(f"k = {k} -> các giá trị nhỏ x mà x + {k} có mặt:", good)
    return len(good)

print("đáp án:", counter_demo([3, 1, 4, 1, 5], 2))
print("đáp án:", counter_demo([1, 3, 1, 5, 4], 0))

cnt = {3: 1, 1: 2, 4: 1, 5: 1}
k = 2 -> các giá trị nhỏ x mà x + 2 có mặt: [3, 1]
đáp án: 2
cnt = {1: 2, 3: 1, 5: 1, 4: 1}
k = 0 -> các giá trị xuất hiện >= 2 lần: [1]
đáp án: 1


**Code nộp LeetCode (cách 3):**

In [14]:
class Solution3:
    def findPairs(self, nums: List[int], k: int) -> int:
        cnt = Counter(nums)
        if k == 0:
            return sum(1 for c in cnt.values() if c >= 2)
        return sum(1 for x in cnt if x + k in cnt)


for nums, k, expected in examples:
    assert Solution3().findPairs(nums, k) == expected
print("Cách 3 OK")

Cách 3 OK


**Độ phức tạp:** Time `O(n)` (đếm một lượt + duyệt giá trị phân biệt). Space `O(n)` cho `Counter`.

### 7.3. Cách 4 — Sắp xếp + hai con trỏ

**Ý tưởng khác:** không dùng bảng băm. Sắp xếp `nums` tăng dần. Với mỗi **giá trị phân biệt** `nums[i]`, ta cần biết có `nums[j]` (với `j > i`) bằng `nums[i] + k` hay không. Khi `i` tăng (giá trị không giảm) thì mục tiêu `nums[i] + k` cũng không giảm, nên con trỏ `j` **chỉ cần đi tới**, không quay lại.

**Giải thích từng đoạn:**

1. `nums.sort()` (làm trên bản sao nếu không muốn phá input; LeetCode cho phép sắp xếp tại chỗ).
2. Duyệt `i` từ trái sang phải; nếu `nums[i] == nums[i - 1]` thì **bỏ qua** (giá trị trùng đã xét — tránh đếm cặp hai lần).
3. `j = max(j, i + 1)`: bảo đảm `j > i` (hai vị trí khác nhau). Đây là chỗ xử lý `k = 0` — với `k = 0`, `j = i + 1` sẽ kiểm tra `nums[i + 1] == nums[i]`, tức giá trị có lặp hay không.
4. Tăng `j` khi `nums[j] - nums[i] < k`. Khi dừng: nếu `j < n` và hiệu `== k` thì có một cặp → `ans += 1`.

**Code demo:**

In [15]:
def two_pointers_demo(nums, k):
    nums = sorted(nums)
    n = len(nums)
    print("sorted:", nums)
    ans = 0
    j = 0
    for i in range(n):
        if i > 0 and nums[i] == nums[i - 1]:
            print(f"  i = {i}: {nums[i]} trùng -> bỏ qua")
            continue
        j = max(j, i + 1)
        while j < n and nums[j] - nums[i] < k:
            j += 1
        if j < n and nums[j] - nums[i] == k:
            ans += 1
            print(f"  i = {i}, j = {j}: {nums[j]} - {nums[i]} == {k} -> cặp ({nums[i]}, {nums[j]}), ans = {ans}")
        else:
            print(f"  i = {i}, j = {j}: không có cặp với {nums[i]}")
    return ans

print("đáp án:", two_pointers_demo([3, 1, 4, 1, 5], 2))

sorted: [1, 1, 3, 4, 5]
  i = 0, j = 2: 3 - 1 == 2 -> cặp (1, 3), ans = 1
  i = 1: 1 trùng -> bỏ qua
  i = 2, j = 4: 5 - 3 == 2 -> cặp (3, 5), ans = 2
  i = 3, j = 5: không có cặp với 4
  i = 4, j = 5: không có cặp với 5
đáp án: 2


**Code nộp LeetCode (cách 4):**

In [16]:
class Solution4:
    def findPairs(self, nums: List[int], k: int) -> int:
        nums.sort()
        n = len(nums)
        ans = 0
        j = 0
        for i in range(n):
            if i > 0 and nums[i] == nums[i - 1]:
                continue
            j = max(j, i + 1)
            while j < n and nums[j] - nums[i] < k:
                j += 1
            if j < n and nums[j] - nums[i] == k:
                ans += 1
        return ans


for nums, k, expected in examples:
    assert Solution4().findPairs(list(nums), k) == expected
print("Cách 4 OK")

Cách 4 OK


**Độ phức tạp:** Time `O(n log n)` cho sắp xếp + `O(n)` cho hai con trỏ (mỗi con trỏ chỉ đi tới, `j` tối đa `n` bước tổng cộng) ⇒ `O(n log n)`. Space `O(1)` thêm (không tính bộ nhớ của `sort`; Python dùng Timsort cần tới `O(n)` bộ nhớ phụ trong trường hợp xấu). Ưu điểm: không dùng bảng băm; nhược điểm: sửa input (có thể tránh bằng `sorted`).

### 7.4. Cách 5 — Sắp xếp + tìm kiếm nhị phân

**Ý tưởng khác:** vẫn sắp xếp, nhưng thay con trỏ `j` bằng **tìm nhị phân** `nums[i] + k` trong phần bên phải `i`.

**Giải thích từng đoạn:**

1. Sắp xếp `nums`.
2. Với mỗi giá trị phân biệt `nums[i]` (bỏ qua giá trị trùng liền trước), tìm `p = bisect_left(nums, nums[i] + k, i + 1)` — vị trí đầu tiên từ `i + 1` trở đi có giá trị `>= nums[i] + k`.
3. Nếu `p < n` và `nums[p] == nums[i] + k` → có một cặp.
4. Tìm từ `i + 1` (`lo = i + 1`) bảo đảm hai vị trí khác nhau. Với `k = 0`: ta tìm chính giá trị `nums[i]` ở các vị trí sau `i` — tức giá trị có lặp lại hay không.

**Code demo:**

In [17]:
def bsearch_demo(nums, k):
    nums = sorted(nums)
    n = len(nums)
    print("sorted:", nums)
    ans = 0
    for i in range(n):
        if i > 0 and nums[i] == nums[i - 1]:
            continue
        target = nums[i] + k
        p = bisect_left(nums, target, i + 1)
        found = p < n and nums[p] == target
        print(f"  i = {i}, x = {nums[i]}: tìm {target} từ vị trí {i + 1} -> p = {p}, {'có' if found else 'không'}")
        ans += found
    return ans

print("đáp án:", bsearch_demo([1, 3, 1, 5, 4], 0))
print("đáp án:", bsearch_demo([1, 2, 3, 4, 5], 1))

sorted: [1, 1, 3, 4, 5]
  i = 0, x = 1: tìm 1 từ vị trí 1 -> p = 1, có
  i = 2, x = 3: tìm 3 từ vị trí 3 -> p = 3, không
  i = 3, x = 4: tìm 4 từ vị trí 4 -> p = 4, không
  i = 4, x = 5: tìm 5 từ vị trí 5 -> p = 5, không
đáp án: 1
sorted: [1, 2, 3, 4, 5]
  i = 0, x = 1: tìm 2 từ vị trí 1 -> p = 1, có
  i = 1, x = 2: tìm 3 từ vị trí 2 -> p = 2, có
  i = 2, x = 3: tìm 4 từ vị trí 3 -> p = 3, có
  i = 3, x = 4: tìm 5 từ vị trí 4 -> p = 4, có
  i = 4, x = 5: tìm 6 từ vị trí 5 -> p = 5, không
đáp án: 4


**Code nộp LeetCode (cách 5):**

In [18]:
class Solution5:
    def findPairs(self, nums: List[int], k: int) -> int:
        nums.sort()
        n = len(nums)
        ans = 0
        for i in range(n):
            if i > 0 and nums[i] == nums[i - 1]:
                continue
            target = nums[i] + k
            p = bisect_left(nums, target, i + 1)
            if p < n and nums[p] == target:
                ans += 1
        return ans


for nums, k, expected in examples:
    assert Solution5().findPairs(list(nums), k) == expected
print("Cách 5 OK")

Cách 5 OK


**Độ phức tạp:** Time `O(n log n)` (sắp xếp + tối đa `n` lần tìm nhị phân, mỗi lần `O(log n)`). Space `O(1)` thêm (ngoài bộ nhớ sắp xếp). Cách này dễ nghĩ nhưng chậm hơn hai con trỏ về hằng số; có lợi khi mảng đã được sắp xếp sẵn và bạn chỉ cần tra cứu.

## 8. TRACE từng bước

### 8.1. Trace cách 2 (một lượt, hai `set`)

Bảng trạng thái với `nums = [3, 1, 4, 1, 5]`, `k = 2`:

| Cột | Ý nghĩa |
|---|---|
| `x` | số đang duyệt |
| `x-k in vis?` | tra cặp có `x` là giá trị lớn |
| `x+k in vis?` | tra cặp có `x` là giá trị nhỏ |
| `vis` | tập đã thấy (sau khi thêm `x`) |
| `ans` | tập giá trị nhỏ của các cặp đã tìm thấy |

In [19]:
def trace_hashset(nums, k):
    vis, ans = set(), set()
    print(f"nums = {nums}, k = {k}")
    for step, x in enumerate(nums, 1):
        a = (x - k) in vis
        b = (x + k) in vis
        if a:
            ans.add(x - k)
        if b:
            ans.add(x)
        vis.add(x)
        print(f"bước {step}: x = {x:>2} | x-k={x - k:>2} trong vis? {str(a):5} | x+k={x + k:>2} trong vis? {str(b):5}"
              f" | vis = {sorted(vis)} | ans = {sorted(ans)}")
    print("=> đáp án", len(ans))
    return len(ans)

trace_hashset([3, 1, 4, 1, 5], 2)

nums = [3, 1, 4, 1, 5], k = 2
bước 1: x =  3 | x-k= 1 trong vis? False | x+k= 5 trong vis? False | vis = [3] | ans = []
bước 2: x =  1 | x-k=-1 trong vis? False | x+k= 3 trong vis? True  | vis = [1, 3] | ans = [1]
bước 3: x =  4 | x-k= 2 trong vis? False | x+k= 6 trong vis? False | vis = [1, 3, 4] | ans = [1]
bước 4: x =  1 | x-k=-1 trong vis? False | x+k= 3 trong vis? True  | vis = [1, 3, 4] | ans = [1]
bước 5: x =  5 | x-k= 3 trong vis? True  | x+k= 7 trong vis? False | vis = [1, 3, 4, 5] | ans = [1, 3]
=> đáp án 2


2

Quan sát: ở bước 2 (`x = 1`), `x + k = 3` đã có trong `vis` nên `ans` nhận `1` (giá trị nhỏ của cặp `(1,3)`). Ở bước 4 `x = 1` lại xuất hiện, `x + k = 3` vẫn có → `ans.add(1)` nhưng `1` đã có sẵn nên **không tăng** (nhờ `set`, cặp không bị đếm hai lần). Ở bước 5 (`x = 5`), `x - k = 3` có trong `vis` → `ans.add(3)`.

### 8.2. Trace cách 3 (`Counter`) với `k = 0`

Ta chạy trên `nums = [1, 3, 1, 5, 4, 4, 4]` để thấy hai trường hợp: giá trị lặp (`1` hai lần, `4` ba lần) và giá trị đứng một mình (`3`, `5`).

In [20]:
def trace_counter(nums, k):
    cnt = Counter(nums)
    print(f"nums = {nums}, k = {k}\ncnt = {dict(cnt)}")
    ans = 0
    for x in cnt:
        if k == 0:
            ok = cnt[x] >= 2
            print(f"x = {x}: xuất hiện {cnt[x]} lần -> {'cặp (x, x)' if ok else 'không có cặp'}")
        else:
            ok = (x + k) in cnt
            print(f"x = {x}: {x + k} có mặt? {ok}")
        ans += ok
    print("=> đáp án", ans)
    return ans

trace_counter([1, 3, 1, 5, 4, 4, 4], 0)
trace_counter([1, 3, 1, 5, 4, 4, 4], 1)

nums = [1, 3, 1, 5, 4, 4, 4], k = 0
cnt = {1: 2, 3: 1, 5: 1, 4: 3}
x = 1: xuất hiện 2 lần -> cặp (x, x)
x = 3: xuất hiện 1 lần -> không có cặp
x = 5: xuất hiện 1 lần -> không có cặp
x = 4: xuất hiện 3 lần -> cặp (x, x)
=> đáp án 2
nums = [1, 3, 1, 5, 4, 4, 4], k = 1
cnt = {1: 2, 3: 1, 5: 1, 4: 3}
x = 1: 2 có mặt? False
x = 3: 4 có mặt? True
x = 5: 6 có mặt? False
x = 4: 5 có mặt? True
=> đáp án 2


2

### 8.3. Trace cách 4 (sắp xếp + hai con trỏ), kèm một ví dụ có số âm

Ta in vị trí của `i`, `j` và hiệu `nums[j] - nums[i]` ở **mỗi lần** `j` tăng, để thấy `j` chỉ đi tới.

In [21]:
def trace_two_pointers(nums, k):
    nums = sorted(nums)
    n = len(nums)
    print(f"sorted = {nums}, k = {k}")
    ans, j = 0, 0
    for i in range(n):
        if i > 0 and nums[i] == nums[i - 1]:
            print(f"i = {i}: nums[i] = {nums[i]} trùng liền trước -> bỏ qua")
            continue
        j = max(j, i + 1)
        print(f"i = {i}: nums[i] = {nums[i]}, bắt đầu với j = {j}")
        while j < n and nums[j] - nums[i] < k:
            print(f"    j = {j}: {nums[j]} - ({nums[i]}) = {nums[j] - nums[i]} < {k} -> j += 1")
            j += 1
        if j < n and nums[j] - nums[i] == k:
            ans += 1
            print(f"    j = {j}: hiệu == {k} -> CẶP ({nums[i]}, {nums[j]}), ans = {ans}")
        else:
            print(f"    j = {j}: không tìm thấy giá trị {nums[i] + k}")
    print("=> đáp án", ans)
    return ans

trace_two_pointers([-3, 0, -1, 2, 2, 0, -1], 2)

sorted = [-3, -1, -1, 0, 0, 2, 2], k = 2
i = 0: nums[i] = -3, bắt đầu với j = 1
    j = 1: hiệu == 2 -> CẶP (-3, -1), ans = 1
i = 1: nums[i] = -1, bắt đầu với j = 2
    j = 2: -1 - (-1) = 0 < 2 -> j += 1
    j = 3: 0 - (-1) = 1 < 2 -> j += 1
    j = 4: 0 - (-1) = 1 < 2 -> j += 1
    j = 5: không tìm thấy giá trị 1
i = 2: nums[i] = -1 trùng liền trước -> bỏ qua
i = 3: nums[i] = 0, bắt đầu với j = 5
    j = 5: hiệu == 2 -> CẶP (0, 2), ans = 2
i = 4: nums[i] = 0 trùng liền trước -> bỏ qua
i = 5: nums[i] = 2, bắt đầu với j = 6
    j = 6: 2 - (2) = 0 < 2 -> j += 1
    j = 7: không tìm thấy giá trị 4
i = 6: nums[i] = 2 trùng liền trước -> bỏ qua
=> đáp án 2


2

Sau sắp xếp: `[-3, -1, -1, 0, 0, 2, 2]`. Các giá trị phân biệt là `-3, -1, 0, 2`: `-3` cần `-1` (có, cặp `(-3,-1)`); `-1` cần `1` (không có); `0` cần `2` (có, cặp `(0,2)`); `2` cần `4` (không có). Đáp án `2`. Các bản sao của `-1`, `0`, `2` bị bỏ qua nhờ bước "trùng liền trước", và `j` chỉ đi tới (không bao giờ lùi).

## 9. Code nộp LeetCode

Chọn **cách 2 (một lượt, hai `set`)**: `O(n)`, ngắn gọn, không sửa input. Đây là code đúng class/method/signature:

In [22]:
from typing import List


class Solution:
    def findPairs(self, nums: List[int], k: int) -> int:
        ans = set()
        vis = set()
        for x in nums:
            if x - k in vis:
                ans.add(x - k)
            if x + k in vis:
                ans.add(x)
            vis.add(x)
        return len(ans)


for nums, k, expected in examples:
    got = Solution().findPairs(nums, k)
    print(got, "(kỳ vọng", expected, ")")
    assert got == expected

2 (kỳ vọng 2 )
4 (kỳ vọng 4 )
1 (kỳ vọng 1 )


## 10. Vì sao thuật toán đúng

**Bước 1: quy về giá trị nhỏ.** Một cặp k-diff `{a, b}` với `|a - b| = k` được xác định hoàn toàn bởi **giá trị nhỏ** `x = min(a, b)` (giá trị lớn là `x + k`). Hai cặp chỉ số cho cùng `x` là cùng một cặp giá trị. Do đó số cặp duy nhất = số giá trị `x` "tốt" — tức tồn tại hai **vị trí khác nhau** `i != j` sao cho `nums[i] = x`, `nums[j] = x + k`.

**Cách 2 (hai `set`).** Bất biến: sau khi xử lý các phần tử `nums[0..t]`, `vis` = tập giá trị của chúng và `ans` = tập các `x` "tốt" **chỉ dùng các phần tử trong tiền tố này**. Khi thêm `nums[t + 1] = y`:

- Mọi cặp mới (chưa có trong tiền tố) phải chứa `y` và một phần tử `z` ở tiền tố với `|y - z| = k`; tức `z = y - k` (khi đó `y` là giá trị lớn, giá trị nhỏ là `y - k`) hoặc `z = y + k` (khi đó giá trị nhỏ là `y`).
- Code kiểm tra đúng hai khả năng đó trong `vis` (là tập giá trị tiền tố) và thêm đúng giá trị nhỏ vào `ans`. Vì `ans` là `set` nên giá trị đã có không tính lại.

Khi `k = 0`: hai khả năng trùng nhau `z = y`; điều kiện `y in vis` nghĩa là `y` đã xuất hiện ở vị trí **trước** → hai vị trí khác nhau, hợp lệ. `vis.add(y)` làm sau cùng nên lần xuất hiện đầu của `y` không tự ghép với chính nó. Kết thúc, `ans` chứa mọi `x` tốt, `len(ans)` là đáp án.

**Cách 3 (`Counter`).** `x` tốt với `k > 0` ⇔ `x` và `x + k` đều có mặt (khác nhau nên nằm ở hai vị trí khác nhau) ⇔ `x in cnt` và `x + k in cnt`. Với `k = 0`: cần hai vị trí cùng giá trị ⇔ `cnt[x] >= 2`. Đếm các `x` phân biệt nên không trùng.

**Cách 4, 5 (sắp xếp).** Sau sắp xếp, các giá trị bằng nhau liền kề; xét mỗi giá trị phân biệt `x = nums[i]` (đại diện là phần tử **đầu tiên** của nhóm, nên mọi bản sao còn lại của `x` nằm ở vị trí `> i`). Cần có `j > i` với `nums[j] = x + k`: nếu `k > 0` thì `j` nằm sau cả nhóm của `x`; nếu `k = 0` thì `j` là một bản sao khác của `x` — đều thuộc `j > i`. Hai con trỏ: mục tiêu `x + k` không giảm khi `i` tăng nên `j` không phải quay lại; tìm nhị phân từ `i + 1` tìm đúng giá trị nhỏ nhất `>= x + k` và kiểm tra bằng.

**Vét cạn.** Là bản dịch nguyên văn của định nghĩa nên đúng.

## 11. Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn mọi cặp chỉ số | `O(n^2)` | `O(p)` |
| 2. Một lượt + hai `set` (nộp) | `O(n)` | `O(n)` |
| 3. `Counter` | `O(n)` | `O(n)` |
| 4. Sắp xếp + hai con trỏ | `O(n log n)` | `O(1)` thêm (tính bộ nhớ sort thì `O(n)`) |
| 5. Sắp xếp + tìm nhị phân | `O(n log n)` | `O(1)` thêm (tính bộ nhớ sort thì `O(n)`) |

`p` là số cặp phân biệt (≤ `n`). Với `n = 10^4`: cách 2 chỉ cần cỡ `2 * 10^4` phép tra; vét cạn cần gần `5 * 10^7` phép so sánh. Đo thực tế:

In [23]:
rng = random.Random(532)
big = [rng.randint(-10**7, 10**7) for _ in range(10**4)]
dup = [rng.randint(-50, 50) for _ in range(10**4)]     # nhiều giá trị lặp

for name, arr, k in (("ngẫu nhiên rộng", big, 1000), ("nhiều giá trị lặp", dup, 3), ("k = 0", dup, 0)):
    results = {}
    for label, f in (("cách 2", Solution2().findPairs), ("cách 3", Solution3().findPairs),
                     ("cách 4", Solution4().findPairs), ("cách 5", Solution5().findPairs)):
        a = list(arr)
        t0 = time.perf_counter()
        results[label] = (f(a, k), (time.perf_counter() - t0) * 1000)
    answers = {v[0] for v in results.values()}
    assert len(answers) == 1
    print(f"{name:18} k = {k:>4}: đáp án {answers.pop():>5} | " +
          " | ".join(f"{lb} {ms:5.1f} ms" for lb, (_, ms) in results.items()))

ngẫu nhiên rộng    k = 1000: đáp án     3 | cách 2   1.7 ms | cách 3   1.2 ms | cách 4   3.6 ms | cách 5   3.7 ms
nhiều giá trị lặp  k =    3: đáp án    98 | cách 2   1.3 ms | cách 3   0.4 ms | cách 4   3.2 ms | cách 5   1.9 ms
k = 0              k =    0: đáp án   101 | cách 2   1.4 ms | cách 3   0.5 ms | cách 4   1.8 ms | cách 5   1.7 ms


Bốn cách nhanh đều chạy trong vài mili-giây với `n = 10^4`.

## 12. Bộ test

### 12.1. Ví dụ của đề và edge case

Ta viết `check_all` chạy mọi cách trên cùng input (nhớ đưa bản sao vì cách 4, 5 sắp xếp tại chỗ) và đòi tất cả trùng nhau.

In [24]:
solvers = [
    ("vét cạn", find_pairs_brute),
    ("cách 2", Solution2().findPairs),
    ("cách 3", Solution3().findPairs),
    ("cách 4", Solution4().findPairs),
    ("cách 5", Solution5().findPairs),
]


def check_all(nums, k, expected=None):
    results = {name: f(list(nums), k) for name, f in solvers}
    values = set(results.values())
    assert len(values) == 1, (nums, k, results)
    got = values.pop()
    if expected is not None:
        assert got == expected, (nums, k, got, expected)
    return got


for nums, k, expected in examples:
    check_all(nums, k, expected)
print("ví dụ đề: OK")

edge = [
    ([1], 0, 0),                          # một phần tử, k = 0: không thể ghép với chính nó
    ([1], 5, 0),                          # một phần tử, k > 0
    ([1, 1], 0, 1),                       # hai bản sao, k = 0
    ([1, 1, 1, 1], 0, 1),                 # nhiều bản sao vẫn là MỘT cặp (1, 1)
    ([1, 1, 1, 2, 2], 1, 1),              # lặp cả hai phía vẫn một cặp (1, 2)
    ([1, 2], 1, 1),
    ([2, 1], 1, 1),                       # thứ tự trong mảng không quan trọng
    ([1, 2], 2, 0),                       # hiệu thực tế là 1, không phải 2
    ([-1, -2, -3], 1, 2),                 # số âm
    ([-5, 5], 10, 1),                     # qua số 0
    ([0, 0, 0], 0, 1),                    # toàn số 0
    ([10**7, -10**7], 10**7, 0),          # hiệu thực tế là 2e7, lớn hơn k
    ([10**7, 0], 10**7, 1),               # k lớn nhất, hiệu đúng 1e7
    ([1, 3, 5, 7], 2, 3),                 # chuỗi cấp số cộng
    ([1, 3, 1, 5, 4], 0, 1),
    ([1, 2, 4, 4, 3, 3, 0, 9, 2, 3], 3, 2),   # các cặp: (0,3) và (1,4)
]
for nums, k, expected in edge:
    got = check_all(nums, k)
    assert got == expected, (nums, k, got, expected)
print("edge case: OK")

ví dụ đề: OK
edge case: OK


Để chắc chắn các đáp án "tính tay" trong bộ edge case không bị ghi nhầm, ta đối chiếu thêm với **hai oracle độc lập** (một dựa trên `itertools.combinations`, một dựa trên đếm giá trị).

In [25]:
def oracle_pairs(nums, k):
    """Oracle độc lập: đếm các cặp GIÁ TRỊ (a, b) phân biệt qua itertools.combinations."""
    return len({(min(a, b), max(a, b)) for a, b in combinations(nums, 2) if abs(a - b) == k})


def oracle_values(nums, k):
    """Oracle thứ hai, viết theo hướng giá trị: x tốt nếu x và x+k có mặt (k=0 thì cần >= 2 bản)."""
    cnt = Counter(nums)
    return sum(1 for x in cnt if (cnt[x] >= 2 if k == 0 else (x + k) in cnt))


for nums, k, expected in edge:
    o1, o2 = oracle_pairs(nums, k), oracle_values(nums, k)
    flag = "" if o1 == o2 == expected else "   <-- số tay SAI"
    print(f"{str(nums):45} k = {k:>9}: số tay {expected}, oracle {o1}/{o2}{flag}")

[1]                                           k =         0: số tay 0, oracle 0/0
[1]                                           k =         5: số tay 0, oracle 0/0
[1, 1]                                        k =         0: số tay 1, oracle 1/1
[1, 1, 1, 1]                                  k =         0: số tay 1, oracle 1/1
[1, 1, 1, 2, 2]                               k =         1: số tay 1, oracle 1/1
[1, 2]                                        k =         1: số tay 1, oracle 1/1
[2, 1]                                        k =         1: số tay 1, oracle 1/1
[1, 2]                                        k =         2: số tay 0, oracle 0/0
[-1, -2, -3]                                  k =         1: số tay 2, oracle 2/2
[-5, 5]                                       k =        10: số tay 1, oracle 1/1
[0, 0, 0]                                     k =         0: số tay 1, oracle 1/1
[10000000, -10000000]                         k =  10000000: số tay 0, oracle 0/0
[10000000, 0]   

Mọi dòng phải in ra cùng một số ở cả ba cột (số tay, oracle 1, oracle 2); nếu có dòng bị đánh dấu thì đó là bài học: **không tin số tính tay mà không có oracle**.

### 12.2. Phản ví dụ cho các suy nghĩ sai

Ta dựng ba "cách giải sai" nghe có lý rồi tìm input làm chúng sai.

In [26]:
def wrong_count_index_pairs(nums, k):
    """SAI: đếm cặp CHỈ SỐ, không loại cặp giá trị trùng."""
    return sum(1 for i in range(len(nums)) for j in range(i + 1, len(nums)) if abs(nums[i] - nums[j]) == k)


def wrong_set_k_zero(nums, k):
    """SAI: 'x + k in set' luôn đúng khi k = 0 (x tự có mặt)."""
    s = set(nums)
    return sum(1 for x in s if x + k in s)


def wrong_both_sides(nums, k):
    """SAI: xét cả x - k lẫn x + k trên tập giá trị -> mỗi cặp bị đếm hai lần (k > 0)."""
    s = set(nums)
    return sum(1 for x in s if x - k in s) + sum(1 for x in s if x + k in s)


tests = [
    ("lặp giá trị", [3, 1, 4, 1, 5, 1], 2),
    ("k = 0, không lặp", [1, 2, 3], 0),
    ("hai phía", [1, 3, 5], 2),
]
for label, nums, k in tests:
    print(f"{label:18} nums = {nums}, k = {k}: đúng = {oracle_pairs(nums, k)}"
          f" | đếm chỉ số = {wrong_count_index_pairs(nums, k)}"
          f" | set k=0 = {wrong_set_k_zero(nums, k)}"
          f" | hai phía = {wrong_both_sides(nums, k)}")

assert wrong_count_index_pairs([3, 1, 4, 1, 5, 1], 2) != oracle_pairs([3, 1, 4, 1, 5, 1], 2)
assert wrong_set_k_zero([1, 2, 3], 0) != oracle_pairs([1, 2, 3], 0)
assert wrong_both_sides([1, 3, 5], 2) != oracle_pairs([1, 3, 5], 2)

lặp giá trị        nums = [3, 1, 4, 1, 5, 1], k = 2: đúng = 2 | đếm chỉ số = 4 | set k=0 = 2 | hai phía = 4
k = 0, không lặp   nums = [1, 2, 3], k = 0: đúng = 0 | đếm chỉ số = 0 | set k=0 = 3 | hai phía = 6
hai phía           nums = [1, 3, 5], k = 2: đúng = 2 | đếm chỉ số = 2 | set k=0 = 2 | hai phía = 4


- `[3,1,4,1,5,1]`, `k=2`: ba số `1` tạo ra **ba** cặp chỉ số `(1,3)` nhưng chỉ là **một** cặp giá trị.
- `[1,2,3]`, `k=0`: không số nào lặp nên đúng là `0`; cách "set" cho `3` vì mọi `x` đều tự có mặt.
- `[1,3,5]`, `k=2`: cặp `(1,3)` bị đếm từ cả hai phía (`x = 3` thấy `x - 2 = 1`, và `x = 1` thấy `x + 2 = 3`) ⇒ gấp đôi.

### 12.3. Đối chiếu nhiều input ngẫu nhiên (seed cố định)

Sinh mảng ngẫu nhiên **đúng ràng buộc** (độ dài ≥ 1, giá trị trong dải nhỏ để có nhiều giá trị lặp và nhiều cặp khớp, `k >= 0`). So mọi cách với hai oracle.

In [27]:
rng = random.Random(20532)
hits = 0
for _ in range(4000):
    n = rng.randint(1, 14)
    span = rng.choice([2, 3, 5, 10, 30])
    nums = [rng.randint(-span, span) for _ in range(n)]
    k = rng.randint(0, span + 1)
    got = check_all(nums, k)
    assert got == oracle_pairs(nums, k) == oracle_values(nums, k), (nums, k)
    hits += got > 0
print(f"4000 input ngẫu nhiên: mọi cách đồng ý với oracle ({hits} input có đáp án > 0)")

# nhiều input lớn: đối chiếu 4 cách nhanh với oracle_values
fast = [Solution2().findPairs, Solution3().findPairs, Solution4().findPairs, Solution5().findPairs]
for _ in range(60):
    n = rng.randint(1, 10**4)
    span = rng.choice([10, 1000, 10**7])
    nums = [rng.randint(-span, span) for _ in range(n)]
    k = rng.randint(0, min(10**7, 2 * span))
    expected = oracle_values(nums, k)
    for f in fast:
        assert f(list(nums), k) == expected
print("60 input lớn (đến 10^4 phần tử): các cách nhanh đồng ý với oracle")

# cực biên
assert Solution().findPairs([5] * 10**4, 0) == 1
assert Solution().findPairs(list(range(10**4)), 1) == 10**4 - 1
assert Solution().findPairs([-10**7, 10**7] * 10, 10**7) == 0
assert Solution().findPairs([0, 10**7], 10**7) == 1
print("cực biên OK")

4000 input ngẫu nhiên: mọi cách đồng ý với oracle (2679 input có đáp án > 0)


60 input lớn (đến 10^4 phần tử): các cách nhanh đồng ý với oracle
cực biên OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Đếm số cặp chỉ số thay vì cặp duy nhất

**Code sai:**

In [28]:
class Wrong1:
    def findPairs(self, nums, k):
        ans = 0
        for i in range(len(nums)):
            for j in range(i + 1, len(nums)):
                if abs(nums[i] - nums[j]) == k:
                    ans += 1                      # SAI: không loại trùng
        return ans

print("sai :", Wrong1().findPairs([3, 1, 4, 1, 5], 2))
print("đúng:", Solution().findPairs([3, 1, 4, 1, 5], 2))

sai : 3
đúng: 2


**Vì sao sai:** đề yêu cầu cặp **duy nhất**. Hai số `1` cho hai cặp chỉ số `(1,3)` nhưng chỉ là một cặp giá trị. Phải dùng `set` (hoặc đi theo giá trị phân biệt).

### Lỗi 2: Quên trường hợp `k = 0`

In [29]:
class Wrong2:
    def findPairs(self, nums, k):
        s = set(nums)
        return sum(1 for x in s if x + k in s)     # SAI khi k = 0: x + 0 = x luôn có mặt

print("sai :", Wrong2().findPairs([1, 3, 1, 5, 4], 0), "(kỳ vọng 1)")
print("đúng:", Solution().findPairs([1, 3, 1, 5, 4], 0))

sai : 4 (kỳ vọng 1)
đúng: 1


**Vì sao sai:** với `k = 0`, mỗi giá trị luôn "tìm thấy chính nó" nên đếm cả các giá trị chỉ xuất hiện **một lần**. Nhưng đề đòi `i != j`: giá trị phải xuất hiện **ít nhất hai lần**. Cần `Counter` (đếm tần suất) hoặc, trong cách một lượt, thêm vào `vis` **sau** khi tra.

### Lỗi 3: Đếm mỗi cặp hai lần (xét cả `x - k` lẫn `x + k` trên tập giá trị)

In [30]:
class Wrong3:
    def findPairs(self, nums, k):
        s = set(nums)
        ans = 0
        for x in s:
            if x - k in s:
                ans += 1
            if x + k in s:
                ans += 1                          # SAI: cặp (x, x+k) cũng được đếm khi xét x+k - k
        return ans

print("sai :", Wrong3().findPairs([1, 2, 3, 4, 5], 1), "(kỳ vọng 4)")
print("đúng:", Solution().findPairs([1, 2, 3, 4, 5], 1))

sai : 8 (kỳ vọng 4)
đúng: 4


**Vì sao sai:** cặp `(1, 2)` được đếm khi xét `x = 2` (thấy `2 - 1 = 1`) và lại khi xét `x = 1` (thấy `1 + 1 = 2`). Chọn **một phía duy nhất** (luôn tính theo giá trị nhỏ). Trong cách hai `set` ở một lượt, hai phép tra vẫn đúng vì chúng dò **tập đã thấy**, mỗi cặp chỉ lộ ra khi phần tử thứ hai của cặp xuất hiện, và `ans` là `set` nên cặp trùng bị loại.

### Lỗi 4: Hai con trỏ nhưng quên bỏ qua giá trị trùng

In [31]:
class Wrong4:
    def findPairs(self, nums, k):
        nums = sorted(nums)
        n = len(nums)
        ans, j = 0, 0
        for i in range(n):
            # SAI: không bỏ qua nums[i] == nums[i-1]
            j = max(j, i + 1)
            while j < n and nums[j] - nums[i] < k:
                j += 1
            if j < n and nums[j] - nums[i] == k:
                ans += 1
        return ans

print("sai :", Wrong4().findPairs([3, 1, 4, 1, 5], 2), "(kỳ vọng 2)")
print("đúng:", Solution4().findPairs([3, 1, 4, 1, 5], 2))

sai : 3 (kỳ vọng 2)
đúng: 2


**Vì sao sai:** sau sắp xếp `[1, 1, 3, 4, 5]`, cả hai số `1` đều ghép với `3`, nên cặp `(1,3)` bị đếm hai lần. Phải bỏ qua giá trị trùng liền trước để mỗi **giá trị phân biệt** chỉ xét một lần.

### Lỗi 5: Hai con trỏ cho `j` bắt đầu bằng `i` (cho phép ghép phần tử với chính nó)

In [32]:
class Wrong5:
    def findPairs(self, nums, k):
        nums = sorted(nums)
        n = len(nums)
        ans, j = 0, 0
        for i in range(n):
            if i > 0 and nums[i] == nums[i - 1]:
                continue
            j = max(j, i)                          # SAI: phải là i + 1
            while j < n and nums[j] - nums[i] < k:
                j += 1
            if j < n and nums[j] - nums[i] == k:
                ans += 1
        return ans

print("sai :", Wrong5().findPairs([1, 2, 3], 0), "(kỳ vọng 0)")
print("đúng:", Solution4().findPairs([1, 2, 3], 0))

sai : 3 (kỳ vọng 0)
đúng: 0


**Vì sao sai:** nếu `j` được phép bằng `i`, với `k = 0` ta có `nums[i] - nums[i] = 0 == k` — phần tử tự ghép với chính nó, trái với `i != j`. Phải ép `j >= i + 1`.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "đếm các **cặp** có hiệu/tổng bằng một giá trị cho trước" và yêu cầu **duy nhất** → nghĩ tới **chuẩn hóa mỗi cặp bằng một phần tử đại diện** (ở đây là giá trị nhỏ) rồi đếm bằng `set`/`Counter`.
- **Họ Two Sum:** tra "phần bù" (`target - x`, hoặc `x ± k`) trong bảng băm đã xây dần. Mẹo chung: *đừng dò cả mảng để tìm phần bù, hãy tra bảng*.
- **Hai con trỏ** dùng được khi mảng đã sắp xếp và mục tiêu thay đổi đơn điệu.
- **Trường hợp `k = 0`** là ngoại lệ kinh điển của họ bài "hiệu bằng k": hai phần tử **cùng giá trị** phải ở **hai vị trí khác nhau**. Luôn tự thử `k = 0` và mảng một phần tử.
- **Đừng quên "unique":** trước khi code, tự hỏi "hai cách chọn khác nhau có cho cùng một kết quả không?" nếu có thì phải khử trùng bằng `set`, bằng sắp xếp + bỏ qua trùng, hoặc bằng cách đếm theo giá trị phân biệt.
- **Thứ tự kiểm tra rồi mới thêm** (`vis.add(x)` đặt **sau** các phép tra) tránh một phần tử ghép với chính nó.
- **Python:** `Counter` + `in` để tra khóa; `sorted(set(nums))` lấy giá trị phân biệt đã sắp xếp; `bisect_left(a, x, lo)` tìm nhị phân từ vị trí `lo`.

## 15. Biến thể

1. **Đếm cặp chỉ số (không cần duy nhất)** — LeetCode 2006 *Count Number of Pairs With Absolute Difference K*: đếm mọi cặp `i < j` với `|nums[i] - nums[j]| == k`. Dùng `Counter` chạy dần: với mỗi `x`, cộng `cnt[x - k] + cnt[x + k]` (với `k = 0` chỉ cộng `cnt[x]`), rồi `cnt[x] += 1`.
2. **Trả về chính các cặp** thay vì chỉ đếm (xem demo bên dưới).
3. **Two Sum (LeetCode 1)** và **Two Sum II (mảng đã sắp xếp)**: cùng họ "tra phần bù"/"hai con trỏ"; ở đó hiệu thay bằng tổng.
4. **`k` có thể âm:** thay `k` bằng `abs(k)` ở đầu hàm.
5. **Dữ liệu dạng luồng**: cách 2 (một lượt, hai `set`) xử lý được khi số đến lần lượt, không cần biết trước toàn bộ.
6. **Mảng rất lớn không vừa bộ nhớ:** sắp xếp ngoài (external sort) + hai con trỏ trên luồng đã sắp.

Demo biến thể 1, 2 (đối chiếu với vét cạn):

In [33]:
def count_index_pairs(nums, k):
    """Biến thể 1: số cặp chỉ số i < j với |nums[i] - nums[j]| == k (không loại trùng)."""
    cnt = Counter()
    ans = 0
    for x in nums:
        ans += cnt[x] if k == 0 else cnt[x - k] + cnt[x + k]
        cnt[x] += 1
    return ans


def list_pairs(nums, k):
    """Biến thể 2: danh sách các cặp giá trị (nhỏ, lớn), đã sắp xếp."""
    cnt = Counter(nums)
    if k == 0:
        return sorted((x, x) for x, c in cnt.items() if c >= 2)
    return sorted((x, x + k) for x in cnt if x + k in cnt)


print(count_index_pairs([3, 1, 4, 1, 5], 2), "cặp chỉ số  vs  ", Solution().findPairs([3, 1, 4, 1, 5], 2), "cặp duy nhất")
print(list_pairs([3, 1, 4, 1, 5], 2))
print(list_pairs([1, 3, 1, 5, 4], 0))
assert list_pairs([3, 1, 4, 1, 5], 2) == [(1, 3), (3, 5)]
assert list_pairs([1, 3, 1, 5, 4], 0) == [(1, 1)]

rng = random.Random(15)
for _ in range(500):
    nums = [rng.randint(-4, 4) for _ in range(rng.randint(1, 10))]
    k = rng.randint(0, 5)
    brute_idx = sum(1 for i in range(len(nums)) for j in range(i + 1, len(nums)) if abs(nums[i] - nums[j]) == k)
    assert count_index_pairs(nums, k) == brute_idx
    assert list_pairs(nums, k) == sorted(pairs_brute(nums, k))
print("biến thể OK")

3 cặp chỉ số  vs   2 cặp duy nhất
[(1, 3), (3, 5)]
[(1, 1)]
biến thể OK


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | Thử mọi cặp chỉ số, `set` các cặp (min, max) | `O(n^2)` | `O(p)` | Chỉ để kiểm chứng input nhỏ |
| 2. Một lượt + hai `set` | Tra `x - k`, `x + k` trong tập đã thấy; `ans` lưu giá trị nhỏ | `O(n)` | `O(n)` | **Cách nộp**, dữ liệu dạng luồng |
| 3. `Counter` | Đếm tần suất; `k = 0`: `cnt >= 2`; `k > 0`: `x + k in cnt` | `O(n)` | `O(n)` | Ngắn gọn, dễ chứng minh, cần tần suất |
| 4. Sắp xếp + hai con trỏ | `j` chỉ đi tới, bỏ qua giá trị trùng | `O(n log n)` | `O(1)` thêm | Muốn tránh bảng băm, đã có mảng sắp xếp |
| 5. Sắp xếp + `bisect_left` | Tìm `x + k` từ `i + 1` | `O(n log n)` | `O(1)` thêm | Mảng đã sắp, tra cứu nhiều lần |

Về ý tưởng: cách 1 không dùng cấu trúc tra cứu; cách 2 tra theo **tập đã thấy** (online); cách 3 tra theo **tần suất toàn cục**; cách 4 dựa vào **tính đơn điệu** của hai con trỏ; cách 5 dựa vào **tìm kiếm nhị phân**.

In [34]:
# so sánh nhanh trên cùng một input lớn (đối chiếu đáp án và thời gian)
rng = random.Random(16)
arr = [rng.randint(-10**4, 10**4) for _ in range(10**4)]
ref = oracle_values(arr, 7)
for name, f in solvers[1:]:
    a = list(arr)
    t0 = time.perf_counter()
    got = f(a, 7)
    ms = (time.perf_counter() - t0) * 1000
    assert got == ref
    print(f"{name}: đáp án {got}, {ms:6.2f} ms")

cách 2: đáp án 3119,   3.19 ms
cách 3: đáp án 3119,   1.92 ms
cách 4: đáp án 3119,   6.35 ms
cách 5: đáp án 3119,   5.26 ms


## 17. 5 bài tập vận dụng (dễ → khó)

Cell **TODO** chỉ gán `None` hoặc định nghĩa hàm rỗng, **không được gọi** khi chạy notebook. Đáp án ở mục 19.

**Bài 1 (đoán output).** Dự đoán rồi chạy:

In [35]:
print(Solution().findPairs([1, 1, 1, 1], 0))
print(Solution().findPairs([1, 5, 9], 4))
print(Solution().findPairs([4, 4, 7], 3))

1
2
1


**Bài 2 (tính tay).** Với `nums = [1, 1, 1, 2, 2]`: (a) số cặp k-diff duy nhất khi `k = 1`; (b) khi `k = 0`; (c) số cặp **chỉ số** `i < j` với hiệu bằng `1`. Gán vào `ans2a`, `ans2b`, `ans2c` (đang là `None`).

In [36]:
ans2a = None   # TODO
ans2b = None   # TODO
ans2c = None   # TODO

**Bài 3 (điền TODO).** Hoàn thành `zero_pairs(nums)` trả về số giá trị phân biệt xuất hiện **ít nhất hai lần** (đây chính là đáp án khi `k = 0`).

In [37]:
def zero_pairs(nums):
    # TODO: dùng Counter
    pass

**Bài 4 (tự viết lại).** Không nhìn mục 7, tự viết `my_find_pairs(nums, k)` theo cách sắp xếp + hai con trỏ (cách 4).

In [38]:
def my_find_pairs(nums, k):
    # TODO
    pass

**Bài 5 (mini challenge).** (a) Viết `count_pairs_sum(nums, s)`: số cặp giá trị duy nhất `(a, b)` với `a + b == s`, hai phần tử ở hai vị trí khác nhau (ví dụ `[1, 1, 2, 3]`, `s = 4` → chỉ có `(1,3)` vì `(2,2)` cần hai số `2`, đáp án `1`; còn `[1, 3, 2, 2]`, `s = 4` → `(1,3)` và `(2,2)`, đáp án `2`). (b) Viết `smallest_k_with_pair(nums)`: giá trị `k >= 0` **nhỏ nhất** sao cho `findPairs(nums, k) >= 1` (mảng có ít nhất 2 phần tử); gợi ý nhỏ: nó là hiệu nhỏ nhất giữa hai phần tử kề nhau sau khi sắp xếp.

In [39]:
def count_pairs_sum(nums, s):
    # TODO
    pass

def smallest_k_with_pair(nums):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):**
- Bài 1: với `k = 0` cần giá trị lặp; với `k > 0` cần tìm `x` và `x + k`.
- Bài 2: liệt kê các giá trị phân biệt `1, 2` và tần suất của chúng (`1` ba lần, `2` hai lần).
- Bài 3: đếm tần suất bằng `Counter`, rồi đếm các tần suất `>= 2`.
- Bài 4: nhớ sắp xếp, bỏ qua trùng, `j` bắt đầu từ `i + 1` và không quay lại.
- Bài 5: (a) với mỗi giá trị `x`, phần bù là `s - x`; nhớ trường hợp `x == s - x`. (b) nghĩ tới hiệu hai số liền kề trong mảng đã sắp xếp.

**Mức 2 (vừa):**
- Bài 2: (a) cặp giá trị duy nhất `(1,2)` → 1. (b) giá trị lặp: `1` (3 lần) và `2` (2 lần) → 2. (c) cặp chỉ số `(1 vị trí, 2 vị trí)` → `3 * 2`.
- Bài 3: `sum(1 for c in cnt.values() if c >= 2)`.
- Bài 4: `j = max(j, i + 1)`; `while j < n and nums[j] - nums[i] < k: j += 1`.
- Bài 5: (a) giữ `Counter`; với `x < s - x` cần `s - x in cnt`; với `x == s - x` cần `cnt[x] >= 2`. (b) `min(b - a for a, b in zip(sorted_nums, sorted_nums[1:]))`.

**Mức 3 (gần đáp án):**
- Bài 3: đáp án nằm ở nhánh `k == 0` của cách 3.
- Bài 4: xem code nộp của cách 4 ở mục 7.3.
- Bài 5(a): chỉ đếm khi `x < s - x` (để mỗi cặp một lần) hoặc `x == s - x` và `cnt[x] >= 2`. (b): sau sắp xếp, hiệu nhỏ nhất giữa hai phần tử kề nhau chính là `k` nhỏ nhất; nếu hai phần tử trùng nhau thì `k = 0`.

## 19. Đáp án tham khảo

Chạy kiểm tra bằng `assert`.

In [40]:
# Bài 1
assert Solution().findPairs([1, 1, 1, 1], 0) == 1
assert Solution().findPairs([1, 5, 9], 4) == 2
assert Solution().findPairs([4, 4, 7], 3) == 1

# Bài 2: nums = [1, 1, 1, 2, 2]
ans2a = 1          # chỉ có cặp giá trị (1, 2)
ans2b = 2          # 1 lặp 3 lần và 2 lặp 2 lần -> (1,1) và (2,2)
ans2c = 6          # 3 cách chọn số 1 x 2 cách chọn số 2
nums2 = [1, 1, 1, 2, 2]
assert ans2a == Solution().findPairs(nums2, 1)
assert ans2b == Solution().findPairs(nums2, 0)
assert ans2c == count_index_pairs(nums2, 1) == sum(1 for i in range(5) for j in range(i + 1, 5) if abs(nums2[i] - nums2[j]) == 1)
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [41]:
# Bài 3
def zero_pairs(nums):
    cnt = Counter(nums)
    return sum(1 for c in cnt.values() if c >= 2)

assert zero_pairs([1, 3, 1, 5, 4]) == 1
assert zero_pairs([1, 2, 3]) == 0
assert zero_pairs([2, 2, 2, 7, 7]) == 2
rng = random.Random(3)
for _ in range(300):
    nums = [rng.randint(-5, 5) for _ in range(rng.randint(1, 12))]
    assert zero_pairs(nums) == Solution().findPairs(nums, 0)
print("Bài 3 OK")

Bài 3 OK


In [42]:
# Bài 4
def my_find_pairs(nums, k):
    nums = sorted(nums)
    n = len(nums)
    ans, j = 0, 0
    for i in range(n):
        if i > 0 and nums[i] == nums[i - 1]:
            continue
        j = max(j, i + 1)
        while j < n and nums[j] - nums[i] < k:
            j += 1
        if j < n and nums[j] - nums[i] == k:
            ans += 1
    return ans

rng = random.Random(4)
for _ in range(500):
    nums = [rng.randint(-8, 8) for _ in range(rng.randint(1, 14))]
    k = rng.randint(0, 9)
    assert my_find_pairs(nums, k) == oracle_pairs(nums, k)
print("Bài 4 OK")

Bài 4 OK


In [43]:
# Bài 5
def count_pairs_sum(nums, s):
    cnt = Counter(nums)
    ans = 0
    for x in cnt:
        y = s - x
        if x < y and y in cnt:
            ans += 1
        elif x == y and cnt[x] >= 2:
            ans += 1
    return ans


def smallest_k_with_pair(nums):
    a = sorted(nums)
    return min(b - x for x, b in zip(a, a[1:]))


assert count_pairs_sum([1, 1, 2, 3], 4) == 1
assert count_pairs_sum([1, 3, 2, 2], 4) == 2
assert count_pairs_sum([5], 10) == 0
assert count_pairs_sum([5, 5], 10) == 1

rng = random.Random(5)
for _ in range(500):
    nums = [rng.randint(-6, 6) for _ in range(rng.randint(1, 12))]
    s = rng.randint(-8, 8)
    brute = len({(min(a, b), max(a, b)) for a, b in combinations(nums, 2) if a + b == s})
    assert count_pairs_sum(nums, s) == brute

assert smallest_k_with_pair([3, 1, 4, 1, 5]) == 0
assert smallest_k_with_pair([1, 5, 9]) == 4
for _ in range(300):
    nums = [rng.randint(-20, 20) for _ in range(rng.randint(2, 10))]
    k = smallest_k_with_pair(nums)
    assert Solution().findPairs(list(nums), k) >= 1
    assert all(Solution().findPairs(list(nums), t) == 0 for t in range(k))
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template một lượt, hai `set` (nộp):**

```python
class Solution:
    def findPairs(self, nums: List[int], k: int) -> int:
        ans, vis = set(), set()
        for x in nums:
            if x - k in vis:
                ans.add(x - k)        # giá trị nhỏ của cặp (x - k, x)
            if x + k in vis:
                ans.add(x)            # giá trị nhỏ của cặp (x, x + k)
            vis.add(x)                # thêm SAU khi tra
        return len(ans)
```

**Template `Counter`:**

```python
cnt = Counter(nums)
if k == 0:
    return sum(1 for c in cnt.values() if c >= 2)
return sum(1 for x in cnt if x + k in cnt)
```

**Template sắp xếp + hai con trỏ:**

```python
nums.sort()
j = 0
for i in range(n):
    if i and nums[i] == nums[i - 1]: continue
    j = max(j, i + 1)
    while j < n and nums[j] - nums[i] < k: j += 1
    if j < n and nums[j] - nums[i] == k: ans += 1
```

**Checklist trước khi submit:**

- [ ] Đã khử trùng (cặp duy nhất): `set`, hoặc xét giá trị phân biệt, hoặc bỏ qua trùng sau sắp xếp?
- [ ] `k = 0` xử lý riêng: giá trị phải xuất hiện **≥ 2 lần**?
- [ ] Mỗi cặp chỉ được đếm **một lần** (chọn một phía: giá trị nhỏ)?
- [ ] Phần tử không ghép với chính nó (`vis.add(x)` đặt sau khi tra; `j` bắt đầu từ `i + 1`)?
- [ ] Thử: một phần tử, toàn giá trị giống nhau, số âm, `k` bằng hiệu lớn nhất, `n = 10^4`?
- [ ] Độ phức tạp `O(n)` hoặc `O(n log n)` đủ cho `n = 10^4` (không dùng `O(n^2)`)?
- [ ] Đã so với vét cạn/oracle trên input nhỏ ngẫu nhiên?